# 04 – Optimizuoto balsavimo ansamblio (Voting Ensemble) eksperimentai

## Darbo tikslas

Šiame Notebook tiriamas dviejų gradientinio stiprinimo modelių ansamblis:

- **GradientBoostingRegressor (GBR)**;
- **HistGradientBoostingRegressor (HGBR)**.

Modelis gerinamas trimis aiškiais etapais:

1. atliekama nedidelė pradinė abiejų bazinių modelių tinklelio paieška;
2. abiejų modelių hiperparametrai išsamiau optimizuojami **Optuna TPE** metodu;
3. iš užfiksuotų geriausių GBR ir HGBR prognozių DEV dalyje parenkamas optimalus Voting svoris.

Pagrindinis modelio parinkimo kriterijus yra **vidutinė DEV MAE**. Papildomai vertinamos RMSE, Peak MAE ir PUR. Visi modeliai vertinami tais pačiais chronologiniais `rolling-origin` skaidiniais, o **TEST šiame Notebook modelio ar hiperparametrų parinkimui nenaudojamas**.

In [ ]:
from pathlib import Path
import sys, warnings, json
warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "hour.csv"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
from src.common import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
work=load_prepare(DATA_PATH)
FEATURES=feature_columns(work)
dev,test=split_dev_test(work)
folds=rolling_origin_folds(dev)

from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from IPython.display import display, Markdown
from scipy.optimize import minimize_scalar
try:
    import optuna
except ImportError:
    %pip install -q optuna
    import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


## 1. Eksperimento korektiškumo patikra

Kad palyginimas su kitais projekto modeliais būtų korektiškas:

- naudojamas tas pats bendras duomenų paruošimas iš `src/common.py`;
- naudojami tie patys DEV ir galutinio TEST laikotarpiai;
- DEV dalyje naudojami tie patys chronologiniai `rolling-origin` skaidiniai;
- hiperparametrai ir Voting svoris parenkami tik pagal DEV;
- `casual` ir `registered` nenaudojami kaip požymiai;
- trūkstamų reikšmių užpildymas kiekviename skaidinyje mokomas tik TRAIN dalyje;
- piko slenkstis nustatomas tik iš konkretaus TRAIN skaidinio;
- naudojama fiksuota `SEED` reikšmė.

Taip galutinį Voting rezultatą vėliau galima vertinti tomis pačiomis sąlygomis kaip MLP, CatBoost ir kitus projekto modelius.

In [ ]:
print("DEV eilučių skaičius:", len(dev))
print("TEST eilučių skaičius:", len(test))
print("Požymių skaičius:", len(FEATURES))
print("Chronologinių DEV skaidinių skaičius:", len(folds))

assert "casual" not in FEATURES
assert "registered" not in FEATURES
assert len(folds) > 0
assert len(test) > 0

print("IŠVADA: pagrindinės eksperimento korektiškumo sąlygos įvykdytos.")
print("TEST modelio ir hiperparametrų parinkimui šiame Notebook nenaudojamas.")

## 2. Voting Ensemble veikimo principas

Dviejų bazinių modelių prognozės sujungiamos svertiniu vidurkiu:

$$
\hat{y}
=
w\hat{y}_{GBR}
+
(1-w)\hat{y}_{HGBR}
$$

čia:

- $\hat{y}$ – galutinė ansamblio prognozė;
- $\hat{y}_{GBR}$ – GBR prognozė;
- $\hat{y}_{HGBR}$ – HGBR prognozė;
- $w$ – GBR svoris;
- $1-w$ – HGBR svoris.

Kode ši taisyklė realizuojama:

```python
pred = w * pred_gbr + (1 - w) * pred_hgbr
```

Kai $w=0$, prognozę sudaro tik HGBR, o kai $w=1$ – tik GBR. Todėl svorio optimizavimas taip pat leidžia patikrinti, ar dviejų modelių sujungimas iš tiesų naudingesnis už vieną bazinį modelį.

## 3. Vertinimo funkcija ir baziniai modeliai

Kiekviena hiperparametrų konfigūracija vertinama visuose DEV `rolling-origin` skaidiniuose. Pagrindinė optimizavimo funkcija:

$$
\theta^*
=
\arg\min_{\theta}
\mathrm{MAE}_{DEV}(\theta)
$$

čia $\theta$ – modelio hiperparametrų rinkinys.

Tokia pati pagrindinė DEV MAE parinkimo logika naudojama ir kituose projekto modeliuose. RMSE, Peak MAE ir PUR nėra ignoruojamos – jos skaičiuojamos galutiniam pasirinktos konfigūracijos vertinimui.

In [ ]:
def make_gbr(n_estimators=300, learning_rate=0.05, max_depth=3,
             min_samples_leaf=1, subsample=1.0):
    return GradientBoostingRegressor(
        n_estimators=int(n_estimators),
        learning_rate=float(learning_rate),
        max_depth=int(max_depth),
        min_samples_leaf=int(min_samples_leaf),
        subsample=float(subsample),
        random_state=SEED
    )

def make_hgbr(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
              l2_regularization=0.0, min_samples_leaf=20):
    return HistGradientBoostingRegressor(
        max_iter=int(max_iter),
        learning_rate=float(learning_rate),
        max_leaf_nodes=int(max_leaf_nodes),
        l2_regularization=float(l2_regularization),
        min_samples_leaf=int(min_samples_leaf),
        random_state=SEED
    )

def eval_model(factory, cfg, return_folds=False):
    rows = []
    for fold_id, (tri, vai) in enumerate(folds, 1):
        tr, va = dev.iloc[tri], dev.iloc[vai]

        imputer = SimpleImputer(strategy="median")
        Xtr = imputer.fit_transform(tr[FEATURES])
        Xv = imputer.transform(va[FEATURES])

        model = factory(**cfg)
        model.fit(Xtr, tr["target"])
        pred = model.predict(Xv)

        peak_threshold = float(tr["target"].quantile(PEAK_Q))
        rows.append({
            "fold": fold_id,
            **metrics(va["target"], pred, peak_threshold)
        })

    df = pd.DataFrame(rows)
    if return_folds:
        return df
    return df[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()

## 4. I etapas – pradinė tinklelio paieška

Pirmasis etapas yra nedidelė tinklelio paieška. Jos paskirtis – gauti aiškų pradinį rezultatą, su kuriuo vėliau galima palyginti išsamesnio optimizavimo naudą.

Tai nėra galutinis modelio optimizavimas.

In [ ]:
gbr_grid = list(ParameterGrid({
    "n_estimators": [200, 400],
    "learning_rate": [0.03, 0.05],
    "max_depth": [2, 3]
}))

hgb_grid = list(ParameterGrid({
    "max_iter": [200, 400],
    "learning_rate": [0.03, 0.05],
    "max_leaf_nodes": [15, 31]
}))

rg = []
for i, cfg in enumerate(gbr_grid, 1):
    print(f"GBR grid {i}/{len(gbr_grid)}: {cfg}")
    r = eval_model(make_gbr, cfg)
    rg.append({**cfg, **r.to_dict()})

rh = []
for i, cfg in enumerate(hgb_grid, 1):
    print(f"HGBR grid {i}/{len(hgb_grid)}: {cfg}")
    r = eval_model(make_hgbr, cfg)
    rh.append({**cfg, **r.to_dict()})

rg = pd.DataFrame(rg).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
rh = pd.DataFrame(rh).sort_values(["MAE", "RMSE"]).reset_index(drop=True)

print("GBR tinklelio paieškos rezultatai:")
display(rg)
print("HGBR tinklelio paieškos rezultatai:")
display(rh)

grid_best_g = {
    "n_estimators": int(rg.iloc[0]["n_estimators"]),
    "learning_rate": float(rg.iloc[0]["learning_rate"]),
    "max_depth": int(rg.iloc[0]["max_depth"])
}
grid_best_h = {
    "max_iter": int(rh.iloc[0]["max_iter"]),
    "learning_rate": float(rh.iloc[0]["learning_rate"]),
    "max_leaf_nodes": int(rh.iloc[0]["max_leaf_nodes"])
}

grid_g_mae = float(rg.iloc[0]["MAE"])
grid_h_mae = float(rh.iloc[0]["MAE"])

display(pd.DataFrame({
    "Modelis": ["GBR – geriausias grid", "HGBR – geriausias grid"],
    "DEV MAE": [grid_g_mae, grid_h_mae]
}))

## 5. II etapas – išsamesnis bazinių modelių optimizavimas

Pradinio tinklelio paieška tikrina tik kelias iš anksto parinktas kombinacijas. Todėl antrame etape naudojamas **Optuna TPE** optimizavimas, kuris kryptingai parenka naujus hiperparametrų variantus pagal ankstesnių bandymų rezultatus.

Abiem modeliams taikoma ta pati eksperimento logika:

- optimizuojama tik DEV dalyje;
- kiekvienas bandymas vertinamas tais pačiais 3 `rolling-origin` skaidiniais;
- tikslo funkcija – vidutinė DEV MAE;
- naudojama fiksuota atsitiktinumo sėkla;
- TEST neliečiamas.

Naudojama po **30 bandymų kiekvienam baziniam modeliui**. Tai išsamesnė paieška už pradinį tinklelį, tačiau nėra teigiama, kad ji įrodo absoliučiai geriausią teoriškai galimą konfigūraciją.

In [ ]:
N_TRIALS_GBR = 30
N_TRIALS_HGBR = 30

def objective_gbr(trial):
    cfg = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 30),
        "subsample": trial.suggest_float("subsample", 0.70, 1.00)
    }
    return float(eval_model(make_gbr, cfg)["MAE"])

def objective_hgbr(trial):
    cfg = {
        "max_iter": trial.suggest_int("max_iter", 150, 700, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 63),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10.0, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 50)
    }
    return float(eval_model(make_hgbr, cfg)["MAE"])

sampler_g = optuna.samplers.TPESampler(seed=SEED)
sampler_h = optuna.samplers.TPESampler(seed=SEED)

study_g = optuna.create_study(direction="minimize", sampler=sampler_g)
study_h = optuna.create_study(direction="minimize", sampler=sampler_h)

print("Optimizuojamas GBR...")
study_g.optimize(objective_gbr, n_trials=N_TRIALS_GBR, show_progress_bar=True)

print("Optimizuojamas HGBR...")
study_h.optimize(objective_hgbr, n_trials=N_TRIALS_HGBR, show_progress_bar=True)

opt_g = study_g.best_params
opt_h = study_h.best_params

opt_g_folds = eval_model(make_gbr, opt_g, return_folds=True)
opt_h_folds = eval_model(make_hgbr, opt_h, return_folds=True)

opt_g_mean = opt_g_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()
opt_h_mean = opt_h_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()

print("Optuna GBR parametrai:", opt_g)
display(opt_g_folds)
print("Optuna HGBR parametrai:", opt_h)
display(opt_h_folds)

## 6. Ar bazinių modelių optimizavimas davė pagerėjimą?

Optuna rezultatai palyginami su pirmojo etapo tinklelio paieška. Tolesnei analizei kiekvienam baziniam modeliui automatiškai pasirenkama ta konfigūracija, kuri pasiekė mažesnę vidutinę DEV MAE.

Todėl išsamesnis optimizavimas nėra laikomas pagerėjimu vien dėl to, kad buvo atliktas – pagerėjimą turi parodyti rezultatas.

In [ ]:
opt_g_mae = float(opt_g_mean["MAE"])
opt_h_mae = float(opt_h_mean["MAE"])

if opt_g_mae < grid_g_mae:
    best_g = opt_g
    best_g_source = "Optuna TPE"
    best_g_mae = opt_g_mae
else:
    best_g = grid_best_g
    best_g_source = "pradinė tinklelio paieška"
    best_g_mae = grid_g_mae

if opt_h_mae < grid_h_mae:
    best_h = opt_h
    best_h_source = "Optuna TPE"
    best_h_mae = opt_h_mae
else:
    best_h = grid_best_h
    best_h_source = "pradinė tinklelio paieška"
    best_h_mae = grid_h_mae

optimization_comparison = pd.DataFrame({
    "Modelis": ["GBR", "GBR", "HGBR", "HGBR"],
    "Etapas": ["Grid", "Optuna TPE", "Grid", "Optuna TPE"],
    "DEV MAE": [grid_g_mae, opt_g_mae, grid_h_mae, opt_h_mae]
})
display(optimization_comparison)

display(Markdown(f'''
### Bazinių modelių parinkimas

- **GBR:** pasirinkta **{best_g_source}**, DEV MAE = **{best_g_mae:.3f}**.
- **HGBR:** pasirinkta **{best_h_source}**, DEV MAE = **{best_h_mae:.3f}**.
'''))

plt.figure(figsize=(9, 4))
labels = ["GBR Grid", "GBR Optuna", "HGBR Grid", "HGBR Optuna"]
values = [grid_g_mae, opt_g_mae, grid_h_mae, opt_h_mae]
plt.bar(labels, values)
plt.ylabel("DEV MAE")
plt.title("Pradinės ir išsamesnės hiperparametrų paieškos palyginimas")
plt.tight_layout()
plt.show()

## 7. III etapas – optimizuotų modelių Voting svorio parinkimas

Užfiksavus geriausią GBR ir HGBR konfigūraciją, kiekviename DEV skaidinyje vieną kartą apskaičiuojamos jų prognozės. Tada ieškoma tokio bendro $w$, kuris minimizuoja vidutinę DEV MAE:

$$
w^*
=
\arg\min_{0\leq w\leq1}
\frac{1}{K}
\sum_{k=1}^{K}
\mathrm{MAE}_k(w)
$$

Šis etapas optimizuoja ne pačių modelių hiperparametrus, o jų indėlį į bendrą prognozę.

In [ ]:
fold_predictions = []

for fold_id, (tri, vai) in enumerate(folds, 1):
    tr, va = dev.iloc[tri], dev.iloc[vai]

    imputer = SimpleImputer(strategy="median")
    Xtr = imputer.fit_transform(tr[FEATURES])
    Xv = imputer.transform(va[FEATURES])

    gbr = make_gbr(**best_g)
    hgbr = make_hgbr(**best_h)
    gbr.fit(Xtr, tr["target"])
    hgbr.fit(Xtr, tr["target"])

    fold_predictions.append({
        "fold": fold_id,
        "y": va["target"].to_numpy(),
        "pred_gbr": gbr.predict(Xv),
        "pred_hgbr": hgbr.predict(Xv),
        "peak_threshold": float(tr["target"].quantile(PEAK_Q)),
        "va": va.copy()
    })

def mean_mae_for_weight(w):
    fold_mae = []
    for pack in fold_predictions:
        pred = w * pack["pred_gbr"] + (1.0 - w) * pack["pred_hgbr"]
        fold_mae.append(np.mean(np.abs(pack["y"] - pred)))
    return float(np.mean(fold_mae))

weight_opt = minimize_scalar(
    mean_mae_for_weight,
    bounds=(0.0, 1.0),
    method="bounded",
    options={"xatol": 1e-4}
)

best_w = float(weight_opt.x)

weight_grid = np.linspace(0, 1, 101)
weight_curve = pd.DataFrame({
    "GBR_svoris": weight_grid,
    "DEV_MAE": [mean_mae_for_weight(w) for w in weight_grid]
})

print(f"Optimalus GBR svoris: {best_w:.4f}")
print(f"Optimalus HGBR svoris: {1-best_w:.4f}")
print(f"Voting DEV MAE: {weight_opt.fun:.3f}")

plt.figure(figsize=(9, 4))
plt.plot(weight_curve["GBR_svoris"], weight_curve["DEV_MAE"])
plt.axvline(best_w, linestyle="--", label=f"Optimalus w = {best_w:.3f}")
plt.xlabel("GBR svoris")
plt.ylabel("DEV MAE")
plt.title("Voting svorio įtaka DEV MAE")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Galutinės Voting konfigūracijos DEV stabilumas

Tas pats užfiksuotas Voting svoris vertinamas visuose chronologiniuose DEV skaidiniuose. Tai parodo ne tik vidutinį rezultatą, bet ir rezultatų kitimą skirtingais laikotarpiais.

In [ ]:
ensemble_rows = []
for pack in fold_predictions:
    pred = best_w * pack["pred_gbr"] + (1.0 - best_w) * pack["pred_hgbr"]
    ensemble_rows.append({
        "fold": pack["fold"],
        **metrics(pack["y"], pred, pack["peak_threshold"])
    })

ensemble_folds = pd.DataFrame(ensemble_rows)
ensemble_mean = ensemble_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()

display(ensemble_folds)
display(pd.DataFrame([ensemble_mean], index=["Vidurkis"]))

display(Markdown(f'''
### Galutinis DEV rezultatas

- DEV MAE = **{ensemble_mean["MAE"]:.3f}**
- DEV RMSE = **{ensemble_mean["RMSE"]:.3f}**
- DEV Peak MAE = **{ensemble_mean["Peak_MAE"]:.3f}**
- DEV PUR = **{ensemble_mean["PUR"]:.3f}**
'''))

## 9. Ar Voting sujungimas naudingesnis už atskirus bazinius modelius?

Ansamblio rezultatas tiesiogiai lyginamas su užfiksuotais GBR ir HGBR modeliais tomis pačiomis DEV sąlygomis.

Santykinis MAE pokytis prieš geriausią atskirą modelį:

$$
\Delta_{\mathrm{MAE}}
=
\frac{
\mathrm{MAE}_{best\ single}
-
\mathrm{MAE}_{Voting}
}{
\mathrm{MAE}_{best\ single}
}
\cdot100\%
$$

Teigiama reikšmė reiškia, kad Voting sumažino MAE. Neigiama reikšmė būtų korektiškas neigiamas eksperimento rezultatas.

In [ ]:
gbr_final_folds = eval_model(make_gbr, best_g, return_folds=True)
hgb_final_folds = eval_model(make_hgbr, best_h, return_folds=True)

gbr_final_mean = gbr_final_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()
hgb_final_mean = hgb_final_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()

single_comparison = pd.DataFrame({
    "GBR": gbr_final_mean,
    "HGBR": hgb_final_mean,
    "Voting Ensemble": ensemble_mean
}).T
display(single_comparison)

best_single_mae = min(float(gbr_final_mean["MAE"]), float(hgb_final_mean["MAE"]))
voting_mae = float(ensemble_mean["MAE"])
improvement_vs_single = (best_single_mae - voting_mae) / best_single_mae * 100

if improvement_vs_single > 0:
    verdict = f"Voting DEV MAE yra {improvement_vs_single:.2f} % mažesnė už geriausio atskiro bazinio modelio MAE."
elif improvement_vs_single < 0:
    verdict = f"Voting DEV MAE yra {abs(improvement_vs_single):.2f} % didesnė už geriausio atskiro bazinio modelio MAE."
else:
    verdict = "Voting ir geriausio atskiro bazinio modelio DEV MAE sutampa."

display(Markdown(f"### Rezultatas\n\n{verdict}"))

## 10. Palyginimas su sezoniniu atskaitos metodu

Naudojamas tas pats sezoninis atskaitos metodas kaip kituose projekto Notebook. Taip Voting rezultatas palyginamas su tuo pačiu pradiniu atskaitos tašku ir tomis pačiomis DEV sąlygomis.

In [ ]:
baseline_rows = []
for fold_id, (tri, vai) in enumerate(folds, 1):
    tr, va = dev.iloc[tri], dev.iloc[vai]
    baseline_pred = seasonal_baseline(tr, va, k_weeks=4)
    peak_threshold = float(tr["target"].quantile(PEAK_Q))
    baseline_rows.append({
        "fold": fold_id,
        **metrics(va["target"], baseline_pred, peak_threshold)
    })

baseline_folds = pd.DataFrame(baseline_rows)
baseline_mean = baseline_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()

baseline_comparison = pd.DataFrame({
    "Sezoninis atskaitos metodas": baseline_mean,
    "Optimizuotas Voting Ensemble": ensemble_mean
}).T
display(baseline_comparison)

mae_change = (baseline_mean["MAE"] - ensemble_mean["MAE"]) / baseline_mean["MAE"] * 100
if mae_change > 0:
    baseline_text = f"Voting DEV MAE yra {mae_change:.1f} % mažesnė už sezoninio atskaitos metodo MAE."
else:
    baseline_text = f"Voting DEV MAE yra {abs(mae_change):.1f} % didesnė už sezoninio atskaitos metodo MAE."

display(Markdown(f"### Rezultatas\n\n{baseline_text}"))

## 11. Faktinės ir prognozuotos paklausos analizė

Toliau analizuojamas paskutinis DEV skaidinys. Šie grafikai naudojami modelio elgsenai paaiškinti, o ne papildomam parametrų parinkimui.

Ideali prognozė tenkintų:

$$
\hat{y}_i=y_i
$$

In [ ]:
last = fold_predictions[-1]
va = last["va"]
y_true = last["y"]
pred = best_w * last["pred_gbr"] + (1.0 - best_w) * last["pred_hgbr"]

plot_n = min(7 * 24, len(va))
time_axis = va["target_time"].iloc[:plot_n] if "target_time" in va.columns else np.arange(plot_n)

plt.figure(figsize=(12, 5))
plt.plot(time_axis, y_true[:plot_n], label="Faktinė paklausa")
plt.plot(time_axis, pred[:plot_n], label="Prognozuota paklausa")
plt.xlabel("Laikas")
plt.ylabel("Dviračių nuomų skaičius")
plt.title("Optimizuotas Voting Ensemble: faktinė ir prognozuota paklausa")
plt.legend()
plt.tight_layout()
plt.show()

lower = min(y_true.min(), pred.min())
upper = max(y_true.max(), pred.max())

plt.figure(figsize=(6, 6))
plt.scatter(y_true, pred, alpha=0.45)
plt.plot([lower, upper], [lower, upper], linestyle="--", label=r"Ideali prognozė: $\hat{y}=y$")
plt.xlabel("Faktinė paklausa")
plt.ylabel("Prognozuota paklausa")
plt.title("Faktinės ir prognozuotos paklausos palyginimas")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Piko valandų ir didžiausių klaidų analizė

Vienos prognozės pasirašyta paklaida:

$$
e_i=y_i-\hat{y}_i
$$

Kai $e_i>0$, paklausa neįvertinta; kai $e_i<0$, paklausa pervertinta.

Kadangi dviračių perskirstymo sprendimams ypač svarbios didelės paklausos valandos, atskirai analizuojamos piko valandos ir didžiausios absoliučios klaidos.

In [ ]:
signed_error = y_true - pred
peak_threshold = last["peak_threshold"]
peak_mask = y_true >= peak_threshold

plt.figure(figsize=(9, 4))
plt.hist(signed_error, bins=30)
plt.axvline(0, linestyle="--", label="Nulinė paklaida")
plt.xlabel(r"Paklaida $e_i=y_i-\hat{y}_i$")
plt.ylabel("Stebėjimų skaičius")
plt.title("Voting prognozavimo paklaidų pasiskirstymas")
plt.legend()
plt.tight_layout()
plt.show()

errors = va.copy()
errors["Faktinė_paklausa"] = y_true
errors["Prognozuota_paklausa"] = pred
errors["Paklaida"] = signed_error
errors["Absoliuti_klaida"] = np.abs(signed_error)
errors["Piko_valanda"] = peak_mask

show_cols = [c for c in [
    "target_time", "hr", "weekday", "workingday", "weathersit",
    "Faktinė_paklausa", "Prognozuota_paklausa",
    "Paklaida", "Absoliuti_klaida", "Piko_valanda"
] if c in errors.columns]

print("10 didžiausių absoliučių klaidų:")
display(errors.nlargest(10, "Absoliuti_klaida")[show_cols])

print("10 didžiausių klaidų piko valandomis:")
display(errors.loc[errors["Piko_valanda"]]
        .nlargest(10, "Absoliuti_klaida")[show_cols])

peak_under_rate = float(np.mean(pred[peak_mask] < y_true[peak_mask])) if peak_mask.any() else np.nan
display(Markdown(f'''
### Paskutinio DEV skaidinio klaidų santrauka

- vidutinė pasirašyta paklaida: **{np.mean(signed_error):.2f}**;
- medianinė pasirašyta paklaida: **{np.median(signed_error):.2f}**;
- piko slenkstis (apskaičiuotas iš TRAIN): **{peak_threshold:.2f}**;
- piko neįvertinimo dalis šiame skaidinyje: **{peak_under_rate:.3f}**.
'''))

## 13. Praktinė reikšmė

Prognozė šiame uždavinyje skirta padėti planuoti dviračių perskirstymą tarp stočių ar aptarnaujamų zonų. Didelė teigiama paklaida reiškia, kad tikroji paklausa buvo didesnė už prognozuotą. Tokie atvejai ypač svarbūs piko metu, nes neįvertinta paklausa gali reikšti per mažą iš anksto suplanuotą dviračių kiekį.

Todėl modelis vertinamas ne tik pagal bendrą MAE ir RMSE, bet ir pagal Peak MAE bei PUR. Galutinis praktinis modelių palyginimas bus atliekamas bendrame TEST etape.

## 14. Galutinės konfigūracijos išsaugojimas

Į `results/best_voting.json` išsaugomi:

- pasirinkti GBR hiperparametrai;
- pasirinkti HGBR hiperparametrai;
- jų parinkimo etapas;
- optimalus Voting svoris.

Šis failas bus naudojamas galutiniame TEST palyginime, kad modelio konfigūracija nebūtų keičiama pamačius TEST rezultatus.

In [ ]:
best_vote = {
    "gbr": best_g,
    "gbr_selection": best_g_source,
    "hgbr": best_h,
    "hgbr_selection": best_h_source,
    "w_gbr": best_w,
    "w_hgbr": 1.0 - best_w,
    "selection_metric": "DEV_MAE"
}

(RESULTS / "best_voting.json").write_text(
    json.dumps(best_vote, indent=2),
    encoding="utf-8"
)

print(json.dumps(best_vote, indent=2))
print("Išsaugota:", RESULTS / "best_voting.json")

## 15. Ribotumai ir šio eksperimento išvada

Hiperparametrų paieška atlikta dviem etapais: pradine tinklelio paieška ir išsamesniu Optuna TPE optimizavimu. Kiekvienam baziniam modeliui pasirinkta mažiausią DEV MAE pasiekusi iš išbandytų konfigūracijų. Po to DEV dalyje optimizuotas jų prognozių sujungimo svoris.

Kadangi tikrinamas baigtinis hiperparametrų variantų skaičius, negalima teigti, kad rasta absoliučiai geriausia teoriškai įmanoma GBR, HGBR ar Voting konfigūracija. Galima teigti tik tai, kad pasirinktas geriausias šiame eksperimente patikrintas variantas pagal nustatytą DEV MAE kriterijų.

Jeigu Voting neaplenkia geriausio atskiro bazinio modelio, tai nėra eksperimento klaida – toks rezultatas rodo, kad šių dviejų modelių prognozių sujungimas pagal pasirinktą procedūrą papildomos naudos nesuteikė.

Galutinė išvada apie Voting Ensemble vietą tarp MLP, CatBoost, N-BEATS ir kitų projekto metodų bus daroma tik bendrame TEST rezultatų palyginime. Šiame Notebook TEST duomenys modelio, hiperparametrų ar svorių parinkimui nenaudojami.